#### Saves 2000 HVG genes and their raw read counts in a matrix

In [1]:
import sys
import os

# Compute project root (go up one level)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))

# Add project root to path
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
sys.path

['/opt/homebrew/Cellar/python@3.10/3.10.17/Frameworks/Python.framework/Versions/3.10/lib/python310.zip',
 '/opt/homebrew/Cellar/python@3.10/3.10.17/Frameworks/Python.framework/Versions/3.10/lib/python3.10',
 '/opt/homebrew/Cellar/python@3.10/3.10.17/Frameworks/Python.framework/Versions/3.10/lib/python3.10/lib-dynload',
 '',
 '/Users/jeromecho/Library/CloudStorage/OneDrive-Personal/Thesis/thesis/code/venvs/scanpy_umap/lib/python3.10/site-packages',
 '/Users/jeromecho/Library/CloudStorage/OneDrive-Personal/Thesis/thesis/code/CellUntangler']

In [4]:
adata = sc.read_h5ad("../../../../data/HGSOC/ALL_CELLS/all_cells_1p.h5ad")

In [5]:
adata = adata.raw.to_adata()

In [6]:
adata.var["gene_symbols"] = adata.var["feature_name"]

In [9]:
cell_cycle_genes_path = "../../../../genes/celluntangler_human_cell_cycle_genes.tsv"
interferon_genes_path = "../../../../genes/HGSOC/multi_signal/jiarui_12_2025/human_interferon_genes.tsv" 
dissociation_genes_path = "../../../../genes/HGSOC/multi_signal/jiarui_12_2025/human_cell_dissociation_genes.tsv" 
# For verification purposes 
pbmsc_cell_type_marker_genes = "../../../../genes/pbmcs_cell_type_marker_genes.tsv" 

cell_cycle_genes = pd.read_csv(cell_cycle_genes_path, header = None, sep="\t")
interferon_genes = pd.read_csv(interferon_genes_path, header = None, sep="\t") # PROGRESS!!!
dissociation_genes = pd.read_csv(dissociation_genes_path, header = None, sep = "\t")
pbmcs_cell_type_genes = pd.read_csv(dissociation_genes_path, header = None, sep = "\t")

In [10]:
cell_cycle_genes_set = set(cell_cycle_genes.iloc[:,0])
interferon_genes_set = set(interferon_genes.iloc[:,0]) 
dissociation_genes_set = set(dissociation_genes.iloc[:,0])
pbmcs_cell_type_genes_set = set(pbmcs_cell_type_genes.iloc[:,0])

In [11]:
contained_genes_cc = adata.var["gene_symbols"].isin(cell_cycle_genes[0])
contained_genes_interferon = adata.var["gene_symbols"].isin(interferon_genes[0])
contained_genes_dissociation = adata.var["gene_symbols"].isin(dissociation_genes[0])

In [12]:
# Save raw counts
adata.layers["counts"] = adata.X.copy()

# Normalize + log for HVG selection
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Compute HVGs
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor="seurat",
    subset=False
)

# Restore raw counts BEFORE slicing
adata.X = adata.layers["counts"]

# Build union of HVGs + marker genes
keep_genes = (
    adata.var["highly_variable"]
    | contained_genes_cc
    | contained_genes_interferon
    | contained_genes_dissociation
)

# Subset
adata = adata[:, keep_genes].copy()

# Optional cleanup
del adata.layers["counts"]


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [28]:
adata.var.index.tolist()

['ENSG00000188290',
 'ENSG00000187608',
 'ENSG00000186891',
 'ENSG00000186827',
 'ENSG00000179403',
 'ENSG00000187730',
 'ENSG00000149527',
 'ENSG00000284703',
 'ENSG00000097021',
 'ENSG00000215788',
 'ENSG00000049249',
 'ENSG00000116285',
 'ENSG00000162444',
 'ENSG00000204624',
 'ENSG00000116663',
 'ENSG00000116670',
 'ENSG00000028137',
 'ENSG00000237301',
 'ENSG00000187144',
 'ENSG00000282843',
 'ENSG00000142623',
 'ENSG00000235185',
 'ENSG00000158747',
 'ENSG00000188257',
 'ENSG00000162545',
 'ENSG00000142798',
 'ENSG00000162552',
 'ENSG00000173372',
 'ENSG00000159189',
 'ENSG00000173369',
 'ENSG00000117318',
 'ENSG00000020633',
 'ENSG00000117632',
 'ENSG00000158022',
 'ENSG00000176083',
 'ENSG00000142748',
 'ENSG00000126709',
 'ENSG00000117748',
 'ENSG00000168528',
 'ENSG00000142910',
 'ENSG00000084636',
 'ENSG00000183615',
 'ENSG00000187513',
 'ENSG00000243749',
 'ENSG00000092853',
 'ENSG00000271554',
 'ENSG00000119535',
 'ENSG00000134690',
 'ENSG00000185668',
 'ENSG00000183682',


In [29]:
adata

AnnData object with n_obs × n_vars = 10199 × 2270
    obs: 'percent.mt', 'percent.rb', 'doublet', 'author_sample_id', 'S.Score', 'G2M.Score', 'Phase', 'CC.Diff', 'author_cell_type', 'nCount_RNA', 'nFeature_RNA', 'doublet_score', 'cell_type_ontology_term_id', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'author_tumor_supersite', 'author_tumor_site', 'author_tumor_subsite', 'author_sort_parameters', 'author_therapy', 'author_procedure', 'author_procedure_type', 'is_primary_data', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'gene_symbols', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'author_cell_type_colors

In [30]:
contained_genes_cc = adata.var["gene_symbols"].isin(cell_cycle_genes[0])

In [47]:
adata.write("../../../../data/HGSOC/ALL_CELLS/all_cells_1p_hvg2000_plus_markers.h5ad")